# 01 - Data Understanding

## Insurance Analytics Platform

This notebook provides the initial exploration of the raw insurance datasets used in the Insurance Analytics Platform project.

The main objectives of this notebook are to:

- Load the raw datasets using reproducible project-relative paths.
- Inspect dataset dimensions, columns, and data types.
- Evaluate missing values and duplicate records.
- Identify potential primary and foreign keys.
- Understand relationships between contracts, claims, and vehicles.
- Detect potential data quality issues before loading the data into MySQL.
- Determine whether the available data can support customer segmentation, claim prediction, claim severity modeling, churn analysis, and RFM-style analysis.

No data cleaning or feature engineering is performed at this stage. The goal is to understand the raw data as it is provided.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Project Paths

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent

else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


print(f"Current directory : {CURRENT_DIR}")
print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Current directory : c:\Users\okand\Desktop\Projects\insurance-analytics-platform\notebooks
Project root      : c:\Users\okand\Desktop\Projects\insurance-analytics-platform
Raw data directory: c:\Users\okand\Desktop\Projects\insurance-analytics-platform\data\raw


In [3]:
assert PROJECT_ROOT.exists(), "Project root directory was not found."
assert RAW_DATA_DIR.exists(), "Raw data directory was not found."

print("Project directories validated successfully.")

Project directories validated successfully.


## 2. Raw Data Files

Before loading the datasets, the files available in the raw data directory are inspected.

This step ensures that the expected source files are available and helps avoid hard-coding assumptions about the filesystem.

In [4]:
raw_files = sorted(RAW_DATA_DIR.glob("*.csv"))

for file_path in raw_files:
    print(file_path.name)

claims.csv
contracts.csv
vehicles.csv


## 3. Loading the Datasets

The raw CSV files are loaded into separate Pandas DataFrames.

In [5]:
CONTRACTS_PATH = RAW_DATA_DIR / "contracts.csv"
CLAIMS_PATH = RAW_DATA_DIR / "claims.csv"
VEHICLES_PATH = RAW_DATA_DIR / "vehicles.csv"

In [6]:
source_files = {
    "contracts": CONTRACTS_PATH,
    "claims": CLAIMS_PATH,
    "vehicles": VEHICLES_PATH,
}

for name, path in source_files.items():
    print(f"{name:<10} -> {path.exists()} | {path.name}")

contracts  -> True | contracts.csv
claims     -> True | claims.csv
vehicles   -> True | vehicles.csv


In [7]:
contracts = pd.read_csv(CONTRACTS_PATH)
claims = pd.read_csv(CLAIMS_PATH)
vehicles = pd.read_csv(VEHICLES_PATH)

### Dataset Loading Validation

After loading the files, the dimensions of each dataset are checked to confirm that the data was read successfully.

In [8]:
datasets = {
    "contracts": contracts,
    "claims": claims,
    "vehicles": vehicles,
}

for name, df in datasets.items():
    print(f"{name:<10}: {df.shape[0]:,} rows × {df.shape[1]} columns")

contracts : 15,000 rows × 14 columns
claims    : 155 rows × 10 columns
vehicles  : 5,390 rows × 10 columns


## 4. Initial Dataset Overview

In [9]:
contracts.head()

,contract_id,client_id,client_name,product,start_date,end_date,annual_premium,status,city_postal,risk_zone,client_age,channel,csp,gender
0,CTR_000001,CLI_000001,Pascal Dubois,Life,11/08/2023,2024-09-08,1974.98€,Renewed,Paris_75001,High,50.00,Agency,NaN,F
1,CTR_000002,CLI_000002,Sophie Simon,Auto,2025-08-12,2026-08-15,€620.93,Active,Bordeaux_33000,Medium,43.00,Phone,Worker,F
2,CTR_000003,CLI_000003,Olivier Durand,Auto,2025-06-14,2026-06-30,$1568.11,Suspended,Paris_75001,High,63.00,Broker,NaN,Male
3,CTR_000004,CLI_000004,Sandrine Michel,Life,04/17/2023,2024-04-20,€1752.17,Renewed,Bordeaux_33000,Medium,54.00,Broker,Manager,Male
4,CTR_000005,CLI_000005,Pierre Durand,Auto,2025-03-02,2026-02-26,$977.59,Suspended,Marseille_13000,Medium,39.00,Web,Employee,M


In [10]:
claims.head()

,claim_id,contract_id,occurrence_date,declaration_date,claim_type,damage_amount,indemnified_amount,status,expert_id,liability
0,CLM_0000001,CTR_008899,26-11-2023,2023-10-02,Theft,15213.03€,10977.27€,Closed,EXP_013,Third_party
1,CLM_0000002,CTR_001770,2025-08-26,2025-08-28,Fire,2321.55€,NaN,Expert_review,EXP_013,Third_party
2,CLM_0000003,CTR_002653,2024-10-25,2024-09-26,Fire,1762.45€,1030.68 EUR,Closed,EXP_001,Third_party
3,CLM_0000004,CTR_002271,2024-04-16,2024-05-27,Collision,3144.06 euros,2119.97€,Closed,NaN,Insured
4,CLM_0000005,CTR_000649,08/03/2025,2025-04-12,Collision,3715.18€,NaN,Expert_review,EXP_007,Third_party


In [11]:
vehicles.head()

,contract_id,brand,model,year,power,fuel_type,current_value,color,usage,previous_claims
0,CTR_000003,BMW,Serie1,"2,022.00",128 HP,Gasoline,29567.77€,Gray,Mixed,0.00
1,CTR_000005,Renault,Megane,"2,024.00",150 HP,Hybrid,14873.92€,Black,Personal,0.00
2,CTR_000012,Peugeot,208,"2,020.00",175,Hybrid,$8362.30,White,Professional,2.00
3,CTR_000015,Renault,Captur,"2,024.00",NaN,Electric,14849.30€,NaN,Mixed,1.00
4,CTR_000016,Renault,Megane,NaN,176hp,Gasoline,€14522.08,Blue,Personal,1.00


## 5. Column Structure

Column names are reviewed before any standardization is applied.

The raw column names are preserved during the data-understanding stage so that potential inconsistencies in the source data can be documented explicitly.

In [12]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-"*60)

    for idx, column in enumerate(df.columns, start=1):
        print(f"{idx:>2}. {column}")


CONTRACTS
------------------------------------------------------------
 1. contract_id
 2. client_id
 3. client_name
 4. product
 5. start_date
 6. end_date
 7. annual_premium
 8. status
 9. city_postal
10. risk_zone
11. client_age
12. channel
13. csp
14. gender

CLAIMS
------------------------------------------------------------
 1. claim_id
 2. contract_id
 3. occurrence_date
 4. declaration_date
 5. claim_type
 6. damage_amount
 7. indemnified_amount
 8. status
 9. expert_id
10. liability

VEHICLES
------------------------------------------------------------
 1. contract_id
 2. brand
 3. model
 4. year
 5. power
 6. fuel_type
 7. current_value
 8. color
 9. usage
10. previous_claims


* contracts.csv: Insurance contracts with client information
* claims.csv: Insurance claims with damage and settlement details
* Vehicle information for auto insurance contracts

## 6. Data Types

The inferred Pandas data types are inspected to identify numeric, categorical, date, and identifier variables.

Unexpected `object` data types may indicate mixed formats, inconsistent values, or columns requiring additional parsing and cleaning.

In [13]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 60)

    display(df.dtypes.rename("dtype").to_frame())


CONTRACTS
------------------------------------------------------------


,dtype
contract_id,str
client_id,str
client_name,str
product,str
start_date,str
end_date,str
annual_premium,str
status,str
city_postal,str
risk_zone,str



CLAIMS
------------------------------------------------------------


,dtype
claim_id,str
contract_id,str
occurrence_date,str
declaration_date,str
claim_type,str
damage_amount,str
indemnified_amount,str
status,str
expert_id,str
liability,str



VEHICLES
------------------------------------------------------------


,dtype
contract_id,str
brand,str
model,str
year,float64
power,str
fuel_type,str
current_value,str
color,str
usage,str
previous_claims,float64


### DataFrame Information

The `DataFrame.info()` method provides a compact overview of column data types, non-null counts, and memory usage.

In [14]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    df.info()


CONTRACTS
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   contract_id     15000 non-null  str    
 1   client_id       15000 non-null  str    
 2   client_name     15000 non-null  str    
 3   product         15000 non-null  str    
 4   start_date      15000 non-null  str    
 5   end_date        15000 non-null  str    
 6   annual_premium  15000 non-null  str    
 7   status          15000 non-null  str    
 8   city_postal     15000 non-null  str    
 9   risk_zone       15000 non-null  str    
 10  client_age      13735 non-null  float64
 11  channel         15000 non-null  str    
 12  csp             13228 non-null  str    
 13  gender          11904 non-null  str    
dtypes: float64(1), str(13)
memory usage: 1.6 MB

CLAIMS
<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 10 columns):
 #   Column         

## 7. Missing Value Analysis

Missing values are evaluated at both count and percentage levels.

Understanding missingness is particularly important in insurance data because missing information may result from operational processes rather than random data loss. The appropriate treatment strategy will therefore be determined only after the business meaning of each variable is understood.

In [15]:
def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_pct": df.isna().mean().mul(100),
            "dtype": df.dtypes
        }
    )

    return (
        summary.query("missing_count > 0").sort_values("missing_pct", ascending=False)
    )

In [16]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(missing_summary(df))


CONTRACTS


,missing_count,missing_pct,dtype
gender,3096,20.64,str
csp,1772,11.81,str
client_age,1265,8.43,float64



CLAIMS


,missing_count,missing_pct,dtype
indemnified_amount,65,41.94,str
liability,47,30.32,str
expert_id,38,24.52,str



VEHICLES


,missing_count,missing_pct,dtype
power,900,16.70,str
color,861,15.97,str
previous_claims,535,9.93,float64
year,273,5.06,float64


## 8. Duplicate Records

Exact duplicate rows are checked before investigating identifier-level duplicates.

Duplicate records may indicate data ingestion issues, repeated transactions, or legitimate multiple observations depending on the business context.

In [17]:
duplicate_summary = pd.DataFrame(
    {
        "dataset": datasets.keys(),
        "rows": [len(df) for df in datasets.values()],
        "duplicate_rows": [
            df.duplicated().sum()
            for df in datasets.values()
        ]
    }
)

duplicate_summary["duplicate_pct"] = (
    duplicate_summary["duplicate_rows"]
    / duplicate_summary["rows"]
    * 100
)

duplicate_summary

,dataset,rows,duplicate_rows,duplicate_pct
0,contracts,15000,0,0.00
1,claims,155,0,0.00
2,vehicles,5390,0,0.00


## 9. Descriptive Statistics

Summary statistics are generated for numeric variables to identify their scale, range, distribution, and potentially suspicious values.

At this stage, unusual values are documented rather than automatically removed because they may represent legitimate insurance observations.

In [18]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")

    numeric_cols = df.select_dtypes(include="number").columns

    if len(numeric_cols) > 0:
        display(df[numeric_cols].describe().T)


CONTRACTS


,count,mean,std,min,25%,50%,75%,max
client_age,"13,735.00",44.44,11.77,18.00,36.00,44.00,53.00,75.00



CLAIMS

VEHICLES


,count,mean,std,min,25%,50%,75%,max
year,"5,117.00","2,019.09",3.80,"2,010.00","2,018.00","2,020.00","2,022.00","2,024.00"
previous_claims,"4,855.00",1.50,1.12,0.00,0.50,1.00,3.00,3.00


## 10. Initial Findings

The initial inspection reveals several important characteristics of the raw datasets:

- The `contracts` dataset contains 15,000 insurance contracts and appears to be the central table of the data model.
- The `claims` dataset contains 155 claim records and is linked to contracts through `contract_id`.
- The `vehicles` dataset contains 5,390 vehicle records and also uses `contract_id` as a potential relationship key.
- No exact duplicate rows were detected in any of the three datasets.
- Several fields that are expected to represent dates or monetary values are currently stored as strings.
- Missing values are present in customer demographic, claim, and vehicle-related attributes.
- Missing values in claim-related fields should not be automatically imputed because they may have a business explanation, such as a rejected, pending, or non-indemnified claim.
- The presence of `client_id` makes customer-level aggregation and segmentation potentially possible.

Before cleaning the data, candidate primary keys, foreign keys, relationship cardinalities, and referential integrity will be examined.

## 11. Candidate Key Analysis

Potential primary and foreign keys are evaluated by checking identifier uniqueness, missing values, and duplicate occurrences.

This step helps determine which columns can serve as primary keys and which columns may represent foreign-key relationships between the datasets.

In [19]:
def key_summary(df: pd.DataFrame, column: str) -> pd.Series:
    return pd.Series(
        {
            "rows": len(df),
            "missing": df[column].isna().sum(),
            "unique_values": df[column].nunique(dropna=False),
            "duplicate_values": df[column].duplicated().sum(),
            "uniqueness_pct":(
                df[column].nunique(dropna=False) / len(df) * 100
            )
        }
    )

In [20]:
key_checks = {
    "contracts.contract_id": key_summary(contracts, "contract_id"),
    "contracts.client_id": key_summary(contracts, "client_id"),
    "claims.claim_id": key_summary(claims, "claim_id"),
    "claims.contract_id": key_summary(claims, "contract_id"),
    "vehicles.contract_id": key_summary(vehicles, "contract_id"),
}

key_summary_df = pd.DataFrame(key_checks).T
key_summary_df

,rows,missing,unique_values,duplicate_values,uniqueness_pct
contracts.contract_id,"15,000.00",0.00,"15,000.00",0.00,100.00
contracts.client_id,"15,000.00",0.00,"15,000.00",0.00,100.00
claims.claim_id,155.00,0.00,155.00,0.00,100.00
claims.contract_id,155.00,0.00,155.00,0.00,100.00
vehicles.contract_id,"5,390.00",0.00,"5,390.00",0.00,100.00


### Key Analysis Findings

The candidate key analysis produced the following findings:

- `contract_id` is unique and non-null in the contracts dataset, making it a strong primary key candidate.
- `claim_id` is unique and non-null in the claims dataset, making it a strong primary key candidate.
- `client_id` is also unique across all contract records. This indicates that, in the current dataset, each client is associated with exactly one contract.
- `contract_id` is unique in both the claims and vehicles datasets, suggesting that each contract is associated with at most one claim record and at most one vehicle record.
- No missing values were detected in any of the identifier columns examined.

Before defining foreign key constraints, referential integrity must be validated to ensure that every `contract_id` in the claims and vehicles datasets exists in the contracts dataset.

## 12. Referential Integrity

Potential foreign key relationships are validated by checking whether every `contract_id` found in the claims and vehicles datasets exists in the contracts dataset.

Records referencing a non-existent contract would represent orphan records and could prevent the creation of reliable foreign key constraints in the relational database.

In [21]:
contract_ids = set(contracts["contract_id"])

claim_contract_ids = set(claims["contract_id"])
vehicle_contract_ids = set(vehicles["contract_id"])

orphan_claim_contracts = claim_contract_ids - contract_ids
orphan_vehicle_contracts = vehicle_contract_ids - contract_ids

print(
    "Claim contract IDs not found in contracts:",
    len(orphan_claim_contracts)
)

print(
    "Vehicle contract IDs not found in contracts:",
    len(orphan_vehicle_contracts)
)

Claim contract IDs not found in contracts: 0
Vehicle contract IDs not found in contracts: 0


### Referential Integrity Findings

The referential integrity check confirmed that all `contract_id` values in the claims and vehicles datasets exist in the contracts dataset.

No orphan records were detected.

This supports the following potential foreign key relationships:

- `claims.contract_id` → `contracts.contract_id`
- `vehicles.contract_id` → `contracts.contract_id`

Since `contract_id` is also unique in both the claims and vehicles datasets, each contract can be associated with at most one claim record and at most one vehicle record in the current dataset.

## 13. Relationship Coverage

The proportion of contracts represented in the claims and vehicles datasets is examined.

This analysis helps determine how much of the insurance portfolio is associated with a claim or a vehicle record and provides additional context for the relational structure of the dataset.

In [23]:
total_contracts = contracts["contract_id"].nunique()

contracts_with_claim = claims["contract_id"].nunique()
contracts_with_vehicle = vehicles["contract_id"].nunique()

coverage_summary = pd.DataFrame(
    {
        "relationship": [
            "Contracts with claims",
            "Contracts with vehicles"
        ],
        "contract_count": [
            contracts_with_claim,
            contracts_with_vehicle
        ]
    }
)

coverage_summary["coverage_pct"] = (
    coverage_summary["contract_count"]
    / total_contracts
    * 100
)

coverage_summary

,relationship,contract_count,coverage_pct
0,Contracts with claims,155,1.03
1,Contracts with vehicles,5390,35.93


### Relationship Coverage Findings

The relationship coverage analysis shows that:

- 155 out of 15,000 contracts (1.03%) are associated with a claim record.
- 5,390 out of 15,000 contracts (35.93%) are associated with a vehicle record.

The very low claim coverage indicates that a future claim-occurrence classification model would involve a highly imbalanced target variable. Therefore, model evaluation should not rely primarily on accuracy. Metrics such as precision, recall, F1-score, and Precision-Recall AUC will be more informative.

Vehicle records are available for approximately one-third of the insurance contracts. This suggests that the vehicles dataset may correspond only to specific insurance products, such as motor-related policies. This assumption will be validated by examining product-level coverage.

## 14. Product-Level Coverage Analysis

The insurance product distribution is examined to determine whether vehicle records are associated with specific product categories.

This analysis will help clarify the business meaning of the vehicles table and determine whether vehicle-related features should be used across the entire portfolio or only for motor insurance contracts.

In [24]:
contracts.head()

,contract_id,client_id,client_name,product,start_date,end_date,annual_premium,status,city_postal,risk_zone,client_age,channel,csp,gender
0,CTR_000001,CLI_000001,Pascal Dubois,Life,11/08/2023,2024-09-08,1974.98€,Renewed,Paris_75001,High,50.00,Agency,NaN,F
1,CTR_000002,CLI_000002,Sophie Simon,Auto,2025-08-12,2026-08-15,€620.93,Active,Bordeaux_33000,Medium,43.00,Phone,Worker,F
2,CTR_000003,CLI_000003,Olivier Durand,Auto,2025-06-14,2026-06-30,$1568.11,Suspended,Paris_75001,High,63.00,Broker,NaN,Male
3,CTR_000004,CLI_000004,Sandrine Michel,Life,04/17/2023,2024-04-20,€1752.17,Renewed,Bordeaux_33000,Medium,54.00,Broker,Manager,Male
4,CTR_000005,CLI_000005,Pierre Durand,Auto,2025-03-02,2026-02-26,$977.59,Suspended,Marseille_13000,Medium,39.00,Web,Employee,M


In [25]:
product_summary = (
    contracts["product"]
    .value_counts(dropna=False)
    .rename_axis("product")
    .reset_index(name="contract_count")
)



product_summary["contract_pct"] = (
    product_summary["contract_count"]
    / len(contracts)
    * 100
)

product_summary

,product,contract_count,contract_pct
0,Auto,6346,42.31
1,Home,4406,29.37
2,Life,2563,17.09
3,Health,1685,11.23


### Product Distribution Findings

The insurance portfolio consists of four product categories:

- Auto insurance represents the largest share of the portfolio, accounting for 42.31% of all contracts.
- Home insurance represents 29.37%.
- Life insurance represents 17.09%.
- Health insurance represents 11.23%.

Since vehicle records are available for 35.93% of all contracts while Auto policies account for 42.31% of the portfolio, the vehicles dataset may primarily represent Auto insurance contracts.

However, this relationship must be explicitly validated before vehicle-related attributes are assumed to be applicable only to Auto policies.

### Vehicle Coverage by Product

Vehicle records are linked to their corresponding insurance contracts to determine which product categories contain vehicle information.

In addition to counting vehicle records by product, the coverage rate within each product is calculated. This helps distinguish between:

- the proportion of vehicle records belonging to each product, and
- the proportion of contracts within each product that have an associated vehicle record.

In [ ]:
vehicle_product = (
    vehicles[["contract_id"]]
    .merge(
        contracts[["contract_id", "product"]],
        on = "contract_id",
        how = "left",
        validate = "one_to_one"
    )
)

vehicle_product["product"].value_counts(dropna=False)

product
Auto    5390
Name: count, dtype: int64

In [31]:
vehicle_product.head(10)

,contract_id,product
0,CTR_000003,Auto
1,CTR_000005,Auto
2,CTR_000012,Auto
3,CTR_000015,Auto
4,CTR_000016,Auto
5,CTR_000018,Auto
6,CTR_000019,Auto
7,CTR_000022,Auto
8,CTR_000023,Auto
9,CTR_000024,Auto


In [27]:
product_summary

,product,contract_count,contract_pct
0,Auto,6346,42.31
1,Home,4406,29.37
2,Life,2563,17.09
3,Health,1685,11.23


In [29]:
vehicle_counts = (
    vehicle_product
    .groupby("product")
    .size()
    .rename("vehicle_contracts")
)

vehicle_coverage_by_product = (
    product_summary
    .set_index("product")
    .join(vehicle_counts)
    .fillna({"vehicle_contracts": 0})
)

vehicle_coverage_by_product["vehicle_contracts"] = (
    vehicle_coverage_by_product["vehicle_contracts"]
    .astype(int)
)

vehicle_coverage_by_product["vehicle_coverage_pct"] = (
    vehicle_coverage_by_product["vehicle_contracts"]
    / vehicle_coverage_by_product["contract_count"]
    * 100
)

vehicle_coverage_by_product.reset_index()

,product,contract_count,contract_pct,vehicle_contracts,vehicle_coverage_pct
0,Auto,6346,42.31,5390,84.94
1,Home,4406,29.37,0,0.00
2,Life,2563,17.09,0,0.00
3,Health,1685,11.23,0,0.00


### Vehicle Coverage Findings

The vehicle coverage analysis confirms that vehicle records are exclusively associated with Auto insurance contracts.

Key findings include:

- All 5,390 vehicle records belong to Auto insurance policies.
- Auto insurance contains 6,346 contracts in total.
- Vehicle information is available for 84.94% of Auto contracts.
- No vehicle records are associated with Home, Life, or Health insurance products.
- Therefore, vehicle-related variables should be treated as Auto-specific attributes rather than general insurance features.

The remaining Auto contracts without vehicle records should be investigated during the data-quality analysis to determine whether the missing vehicle information represents incomplete data or an intentional characteristic of the dataset.

## 15. Claim Coverage by Product

Claim records are linked to their corresponding insurance contracts to examine how claims are distributed across insurance products.

Both the number of claims and the claim coverage rate within each product are calculated. This analysis is important for understanding whether claim occurrence differs substantially between product categories and whether future predictive models should consider product-specific behavior.

In [32]:
claim_product = (
    claims[["claim_id", "contract_id"]]
    .merge(
        contracts[["contract_id", "product"]],
        on="contract_id",
        how="left",
        validate="one_to_one",
    )
)

claim_product.head()

,claim_id,contract_id,product
0,CLM_0000001,CTR_008899,Auto
1,CLM_0000002,CTR_001770,Auto
2,CLM_0000003,CTR_002653,Auto
3,CLM_0000004,CTR_002271,Auto
4,CLM_0000005,CTR_000649,Auto


In [33]:
claim_counts = (
    claim_product
    .groupby("product")
    .size()
    .rename("claim_contracts")
)

claim_coverage_by_product = (
    product_summary
    .set_index("product")
    .join(claim_counts)
    .fillna({"claim_contracts": 0})
)

claim_coverage_by_product["claim_contracts"] = (
    claim_coverage_by_product["claim_contracts"]
    .astype(int)
)

claim_coverage_by_product["claim_coverage_pct"] = (
    claim_coverage_by_product["claim_contracts"]
    / claim_coverage_by_product["contract_count"]
    * 100
)

claim_coverage_by_product.reset_index()

,product,contract_count,contract_pct,claim_contracts,claim_coverage_pct
0,Auto,6346,42.31,136,2.14
1,Home,4406,29.37,19,0.43
2,Life,2563,17.09,0,0.00
3,Health,1685,11.23,0,0.00


### Claim Coverage Findings

The claim coverage analysis reveals substantial differences across insurance products:

- Auto insurance accounts for 136 claim records, corresponding to a claim coverage rate of 2.14%.
- Home insurance accounts for 19 claim records, with a considerably lower claim coverage rate of 0.43%.
- No claim records are observed for Life or Health insurance policies in the current dataset.
- Approximately 87.7% of all observed claims are associated with Auto insurance.

These results indicate that claim occurrence is highly concentrated in the Auto insurance portfolio.

From a predictive modeling perspective, the absence of positive claim observations for Life and Health policies means that claim occurrence cannot be meaningfully learned for these products using the current dataset.

Auto insurance appears to be the strongest candidate for a dedicated claim prediction model because it contains the majority of claim observations and can potentially be enriched with vehicle-specific attributes.

## 16. Vehicle Availability for Auto Claims

Since Auto insurance is the primary candidate for future claim prediction modeling, the availability of vehicle information among Auto claim records is examined.

This analysis determines whether vehicle-specific attributes such as brand, model, vehicle age, power, fuel type, current value, and previous claims can be reliably incorporated into an Auto claim prediction model.

In [34]:
claims.head()

,claim_id,contract_id,occurrence_date,declaration_date,claim_type,damage_amount,indemnified_amount,status,expert_id,liability
0,CLM_0000001,CTR_008899,26-11-2023,2023-10-02,Theft,15213.03€,10977.27€,Closed,EXP_013,Third_party
1,CLM_0000002,CTR_001770,2025-08-26,2025-08-28,Fire,2321.55€,NaN,Expert_review,EXP_013,Third_party
2,CLM_0000003,CTR_002653,2024-10-25,2024-09-26,Fire,1762.45€,1030.68 EUR,Closed,EXP_001,Third_party
3,CLM_0000004,CTR_002271,2024-04-16,2024-05-27,Collision,3144.06 euros,2119.97€,Closed,NaN,Insured
4,CLM_0000005,CTR_000649,08/03/2025,2025-04-12,Collision,3715.18€,NaN,Expert_review,EXP_007,Third_party


In [35]:
auto_claims = (
    claims[["claim_id", "contract_id"]]
    .merge(
        contracts[["contract_id", "product"]],
        on = "contract_id",
        how = "left",
        validate = "one_to_one"
    )
    .query("product == 'Auto'")
    .copy()
)

vehicle_contract_ids = set(vehicles["contract_id"])

auto_claims["has_vehicle_record"] = (
    auto_claims["contract_id"].isin(vehicle_contract_ids)
)

auto_claim_vehicle_summary = (
    auto_claims["has_vehicle_record"]
    .value_counts()
    .rename_axis("has_vehicle_record")
    .reset_index(name="claim_count")
)

auto_claim_vehicle_summary["claim_pct"] = (
    auto_claim_vehicle_summary["claim_count"]
    / len(auto_claims)
    * 100
)

auto_claim_vehicle_summary

,has_vehicle_record,claim_count,claim_pct
0,True,118,86.76
1,False,18,13.24


### Vehicle Availability Findings

Vehicle information is available for 118 of the 136 Auto insurance contracts with a claim, corresponding to a coverage rate of 86.76%.

This coverage rate is slightly higher than the overall vehicle coverage among Auto contracts (84.94%), suggesting that missing vehicle records are not disproportionately concentrated among contracts with claims.

If future claim prediction models are restricted to Auto contracts with available vehicle information, the modeling dataset would contain approximately 5,390 contracts and 118 positive claim observations.

This would allow vehicle-specific features to be incorporated into the model, although the target would remain highly imbalanced.

## 17. Claim Status and Missingness Analysis

Missing values in claim-related variables may reflect legitimate business processes rather than data quality problems.

For example, an indemnified amount may be unavailable because a claim is still under review, rejected, or has not yet been settled.

The relationship between claim status and missing values is therefore examined before any cleaning or imputation decisions are made.

In [36]:
claims.head()

,claim_id,contract_id,occurrence_date,declaration_date,claim_type,damage_amount,indemnified_amount,status,expert_id,liability
0,CLM_0000001,CTR_008899,26-11-2023,2023-10-02,Theft,15213.03€,10977.27€,Closed,EXP_013,Third_party
1,CLM_0000002,CTR_001770,2025-08-26,2025-08-28,Fire,2321.55€,NaN,Expert_review,EXP_013,Third_party
2,CLM_0000003,CTR_002653,2024-10-25,2024-09-26,Fire,1762.45€,1030.68 EUR,Closed,EXP_001,Third_party
3,CLM_0000004,CTR_002271,2024-04-16,2024-05-27,Collision,3144.06 euros,2119.97€,Closed,NaN,Insured
4,CLM_0000005,CTR_000649,08/03/2025,2025-04-12,Collision,3715.18€,NaN,Expert_review,EXP_007,Third_party


In [37]:
claim_status_summary = (
    claims["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="claim_count")
)

claim_status_summary["claim_pct"] = (
    claim_status_summary["claim_count"]
    / len(claims)
    * 100
)

claim_status_summary

,status,claim_count,claim_pct
0,Closed,47,30.32
1,Rejected,43,27.74
2,Expert_review,25,16.13
3,In_progress,23,14.84
4,Open,17,10.97


### Claim Status Findings

The claim status distribution shows that only 30.32% of claim records are closed.

A substantial portion of claims are either rejected or still active in the claim-handling process:

- 27.74% are rejected.
- 16.13% are under expert review.
- 14.84% are in progress.
- 10.97% are open.

This distribution suggests that missing values in fields such as `indemnified_amount`, `expert_id`, and `liability` may be related to the operational status of the claim rather than representing random data quality issues.

Therefore, missing claim-related values should be analyzed conditionally by claim status before any imputation or replacement strategy is considered.

### Indemnified Amount Missingness by Claim Status

The availability of `indemnified_amount` is examined across claim statuses to determine whether missing values are associated with the claim settlement process.

In [38]:
indemnified_missing_by_status = pd.crosstab(
    claims["status"],
    claims["indemnified_amount"].isna(),
    margins=True
)

indemnified_missing_by_status.columns = [
    "indemnified_available",
    "indemnified_missing",
    "total",
]

indemnified_missing_by_status

,indemnified_available,indemnified_missing,total
status,,,
Closed,47,0,47
Expert_review,0,25,25
In_progress,0,23,23
Open,0,17,17
Rejected,43,0,43
All,90,65,155


In [40]:
indemnified_missing_pct = (
    claims
    .assign(
        indemnified_missing=claims["indemnified_amount"].isna()
    )
    .groupby("status")["indemnified_missing"]
    .agg(["sum", "count"])
)

indemnified_missing_pct["missing_pct"] = (
    indemnified_missing_pct["sum"]
    / indemnified_missing_pct["count"]
    * 100
)

indemnified_missing_pct

,sum,count,missing_pct
status,,,
Closed,0,47,0.00
Expert_review,25,25,100.00
In_progress,23,23,100.00
Open,17,17,100.00
Rejected,0,43,0.00


### Indemnified Amount Missingness Findings

Missing values in `indemnified_amount` are strongly associated with claim status.

- All Closed claims contain an indemnified amount.
- All Rejected claims also contain a value in the indemnified amount field.
- All claims classified as Open, In Progress, or Expert Review have a missing indemnified amount.

This pattern indicates that missingness is not random and appears to reflect the claim lifecycle.

However, the presence of a value does not necessarily imply that an indemnity payment was made. In particular, Rejected claims may contain zero-valued indemnification amounts. Therefore, the raw values of `indemnified_amount` must be inspected before interpreting the field or applying any transformation.

### Raw Indemnified Amount Values by Claim Status

Since `indemnified_amount` is currently stored as a string, its raw values are inspected before numeric conversion.

Special attention is given to Rejected claims to determine whether populated values represent actual indemnity payments or zero-valued settlements.

In [41]:
claims.loc[
    claims["status"] == "Rejected",
    ["status", "damage_amount", "indemnified_amount"]
].head(15)

,status,damage_amount,indemnified_amount
9,Rejected,12836.60€,0.00 euros
11,Rejected,1800.12€,0.00€
16,Rejected,411.66€,$0.00
19,Rejected,$1640.40,0.00 EUR
21,Rejected,301.36€,0.00€
34,Rejected,848,0.00€
36,Rejected,2102.55€,0.00€
37,Rejected,250.65€,0.00€
40,Rejected,2853.85€,0.00€
41,Rejected,1015.07 EUR,0.00€


In [42]:
for status in ["closed", "Rejected"]:
    values = (
        claims.loc[
            claims["status"] == status,
            "indemnified_amount"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    print(f"\n{status.upper()}")
    print(values[:20])


CLOSED
<StringArray>
[]
Length: 0, dtype: str

REJECTED
<StringArray>
['0.00 euros', '0.00€', '$0.00', '0.00 EUR', '€0.00', '0']
Length: 6, dtype: str


### Rejected Claim Indemnification Findings

Rejected claims contain explicit zero-valued indemnification amounts rather than missing values.

The same zero amount is represented using several inconsistent string formats, including currency symbols, currency codes, and plain numeric strings.

This indicates that:

- A rejected claim represents a valid zero indemnification rather than a missing payment value.
- Missing `indemnified_amount` values observed in active claim statuses should not be replaced with zero without considering the claim lifecycle.
- Monetary fields require standardized parsing before they can be converted to numeric data types.
- Similar formatting inconsistencies are also visible in `damage_amount`.

In [43]:
closed_mask = claims["status"].eq("Closed")

print("Closed claims:", closed_mask.sum())

print(
    "Closed claims with indemified amount:",
    claims.loc[
        closed_mask,
        "indemnified_amount"
    ].notna().sum()
)

print(
    "Closed claims with missing indemnified amount:",
    claims.loc[
        closed_mask,
        "indemnified_amount"
    ].isna().sum()
)

display(
    claims.loc[
        closed_mask,
        [
            "status",
            "damage_amount",
            "indemnified_amount",
        ]
    ].head(15)
)

Closed claims: 47
Closed claims with indemified amount: 47
Closed claims with missing indemnified amount: 0


,status,damage_amount,indemnified_amount
0,Closed,15213.03€,10977.27€
2,Closed,1762.45€,1030.68 EUR
3,Closed,3144.06 euros,2119.97€
5,Closed,483.92€,$139.34
7,Closed,13896.38€,12720.93€
18,Closed,426.22€,0
20,Closed,5274.03€,4287.87€
22,Closed,8979.43€,8653.23€
23,Closed,19682.88€,15416.79 EUR
32,Closed,2166.96€,1591.69€


### Closed Claim Indemnification Findings

The direct validation confirms that all 47 Closed claims contain a recorded `indemnified_amount`.

However, a Closed claim does not necessarily imply that a positive indemnity payment was made. While most Closed claims contain positive indemnification amounts, some contain explicit zero values.

The observed claim lifecycle can therefore be summarized as follows:

- Rejected claims have a known indemnification amount of zero.
- Open, In Progress, and Expert Review claims do not yet have an indemnification amount recorded.
- Closed claims have a finalized indemnification value, which may be either positive or zero.

This distinction is important for future claim severity modeling. Claim status should not be interpreted as a direct indicator of whether a positive payment occurred.

In addition, both `damage_amount` and `indemnified_amount` contain inconsistent monetary formats and will require standardized parsing during the data-cleaning stage.

### Expert Assignment Missingness by Claim Status

The relationship between claim status and missing `expert_id` values is examined to determine whether expert assignment depends on the stage of the claim-handling process.

Understanding this relationship is important before treating missing expert identifiers as a data quality problem.

In [45]:
expert_missing_by_status = (
    claims
    .assign(
        expert_missing=claims["expert_id"].isna()
    )
    .groupby("status")["expert_missing"]
    .agg(
        missing_count="sum",
        total_claims="count"
    )
)

expert_missing_by_status["missing_pct"] = (
    expert_missing_by_status["missing_count"]
    / expert_missing_by_status["total_claims"]
    * 100
)

expert_missing_by_status

,missing_count,total_claims,missing_pct
status,,,
Closed,13,47,27.66
Expert_review,5,25,20.00
In_progress,4,23,17.39
Open,3,17,17.65
Rejected,13,43,30.23


### Expert Assignment Missingness Findings

Missing `expert_id` values are observed across all claim statuses.

Unlike `indemnified_amount`, expert assignment missingness does not follow a clear deterministic pattern based on claim status:

- 27.66% of Closed claims have no recorded expert.
- 30.23% of Rejected claims have no recorded expert.
- Missingness is also present among Open, In Progress, and Expert Review claims.
- Even 20.00% of claims classified as `Expert_review` do not contain an `expert_id`.

These findings suggest that the absence of an expert identifier cannot be explained solely by the claim lifecycle stage.

At this stage, missing `expert_id` values should therefore be preserved rather than imputed. Further analysis of expert assignment patterns may be performed later if the variable becomes relevant for reporting or predictive modeling.

### Liability Missingness by Claim Status

The relationship between claim status and missing `liability` values is examined to determine whether liability assessment depends on the stage of the claim-handling process.

This analysis will help distinguish between structurally unavailable information and potential data quality issues before any cleaning decisions are made.

In [46]:
liability_missing_by_status = (
    claims
    .assign(
        liability_missing=claims["liability"].isna()
    )
    .groupby("status")["liability_missing"]
    .agg(
        missing_count="sum",
        total_claims="count"
    )
)

liability_missing_by_status["missing_pct"] = (
    liability_missing_by_status["missing_count"]
    / liability_missing_by_status["total_claims"]
    * 100
)

liability_missing_by_status

,missing_count,total_claims,missing_pct
status,,,
Closed,14,47,29.79
Expert_review,5,25,20.00
In_progress,6,23,26.09
Open,3,17,17.65
Rejected,19,43,44.19


### Liability Missingness Findings

Missing `liability` values are present across all claim statuses.

The highest missingness rate is observed among Rejected claims, where 44.19% of records do not contain a liability value. However, missing liability information is also present in Closed, Expert Review, In Progress, and Open claims.

Unlike `indemnified_amount`, liability missingness does not follow a deterministic claim-status pattern.

This suggests that missing `liability` values may reflect incomplete assessment, differences in claim handling, or other operational factors rather than a single lifecycle rule.

Therefore, missing liability values should be preserved at this stage and investigated further before any imputation or replacement strategy is applied.

## 18. Monetary Data Format Inspection

Several monetary variables are currently stored as strings. Before converting them to numeric data types, their raw formatting patterns are inspected.

This step helps identify currency symbols, currency codes, decimal separators, thousands separators, and other inconsistencies that must be handled during data cleaning.

In [47]:
contracts["annual_premium"].dropna().astype(str).head(30).to_list()

['1974.98€',
 '€620.93',
 '$1568.11',
 '€1752.17',
 '$977.59',
 '549.89€',
 '681.51€',
 '579.32€',
 '229.20€',
 '312.56€',
 '273.30€',
 '878.30 EUR',
 '1048.35 euros',
 '$576.80',
 '702.71€',
 '967',
 '$1244.37',
 '732.30€',
 '1242.54€',
 '1495.13€',
 '1692.42€',
 '473.95€',
 '528.27€',
 '744.06€',
 '1957',
 '1128.71€',
 '1875.33€',
 '427.25€',
 '883.02€',
 '1079.66€']

### Annual Premium Format Findings

The `annual_premium` variable is stored as a string despite representing a monetary amount.

Initial inspection reveals several inconsistent representations, including:

- Euro symbols placed before or after the amount
- The `EUR` currency code
- The word `euros`
- Dollar symbols
- Plain numeric strings without any currency indicator

The numeric portion generally appears to use a decimal point, but the presence of multiple currency representations prevents direct conversion to a numeric data type.

Before cleaning the variable, the frequency of each formatting pattern will be examined to determine whether different currency symbols represent actual currencies or simply inconsistent source formatting.

In [48]:
premium_str = (
    contracts["annual_premium"]
    .astype("string")
    .str.strip()
)

premium_format = pd.Series(
    np.select(
        [
            premium_str.str.contains(r"\$", regex=True, na=False),
            premium_str.str.contains("€", regex=False, na=False),
            premium_str.str.contains(r"\bEUR\b", case=False, regex=True, na=False),
            premium_str.str.contains(r"\beuros?\b", case=False, regex=True, na=False),
            premium_str.str.fullmatch(r"\d+(?:\.\d+)?", na=False),
        ],
        [
            "Dollar symbol",
            "Euro symbol",
            "EUR code",
            "Euro word",
            "Plain numeric",
        ],
        default = "Other"
    ),
    index=contracts.index,
    name="premium_format",
)

premium_format.value_counts(dropna=False)

premium_format
Euro symbol      11864
Dollar symbol      818
Euro word          778
EUR code           771
Plain numeric      769
Name: count, dtype: int64

### Annual Premium Format Distribution Findings

All 15,000 `annual_premium` records fall into one of five identifiable formatting patterns.

The Euro symbol is the dominant representation, while smaller portions of the data use the `EUR` currency code, the word `euros`, a dollar symbol, or no currency indicator at all.

No unexpected formatting pattern was detected.

However, the presence of dollar symbols requires further investigation. At this stage, dollar-denominated values should not automatically be treated as Euros or converted using an exchange rate because it is not yet clear whether the symbol represents an actual currency difference or merely inconsistent formatting in the source data.

The numeric distributions across formatting groups will therefore be compared before defining the cleaning rule.

### Annual Premium Numeric Distribution by Format

The numeric component of `annual_premium` is temporarily extracted for exploratory purposes.

Its distribution is compared across formatting groups to determine whether different currency representations correspond to materially different value ranges or are likely to represent inconsistent formatting of the same monetary variable.

The original raw column remains unchanged.

In [49]:
premium_numeric_temp = (
    premium_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

premium_format_analysis = (
    pd.DataFrame(
        {
            "premium_format": premium_format,
            "premium_numeric": premium_numeric_temp,
        }
    )
    .groupby("premium_format")["premium_numeric"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .round(2)
)

premium_format_analysis

,count,mean,median,min,max
premium_format,,,,,
Dollar symbol,818,"1,043.66",902.88,246.50,"2,816.02"
EUR code,771,934.55,767.02,227.28,"2,658.22"
Euro symbol,11864,948.84,790.42,224.14,"2,849.59"
Euro word,778,924.28,788.49,236.61,"2,633.36"
Plain numeric,769,966.27,816.00,225.00,"2,665.00"


In [50]:
print(
    "Unparsed premium values:",
    premium_numeric_temp.isna().sum()
)

Unparsed premium values: 0


### Annual Premium Numeric Distribution Findings

The numeric component of all 15,000 `annual_premium` values was successfully extracted, with no parsing failures.

The distributions across the five formatting groups are broadly similar in terms of minimum, maximum, median, and mean values.

Although records containing a dollar symbol have a slightly higher average premium, their overall value range substantially overlaps with the other formatting groups.

This provides evidence that the different currency representations may reflect formatting inconsistencies rather than genuinely different currencies.

However, before standardizing all values into a single monetary representation, the relationship between formatting patterns and insurance products will be examined to determine whether the observed differences are driven by product composition.

### Annual Premium Format by Insurance Product

The distribution of annual premium formatting patterns is examined across insurance products.

This analysis helps determine whether differences in numeric premium distributions across formatting groups are associated with the underlying product mix rather than actual currency differences.

In [51]:
premium_format_by_product = pd.crosstab(
    contracts["product"],
    premium_format,
    margins=True,
)

premium_format_by_product

premium_format,Dollar symbol,EUR code,Euro symbol,Euro word,Plain numeric,All
product,,,,,,
Auto,328,312,5045,339,322,6346
Health,84,83,1337,81,100,1685
Home,251,246,3466,230,213,4406
Life,155,130,2016,128,134,2563
All,818,771,11864,778,769,15000


In [52]:
premium_format_pct_by_product = (
    pd.crosstab(
        contracts["product"],
        premium_format,
        normalize="index",
    )
    .mul(100)
    .round(2)
)

premium_format_pct_by_product

premium_format,Dollar symbol,EUR code,Euro symbol,Euro word,Plain numeric
product,,,,,
Auto,5.17,4.92,79.50,5.34,5.07
Health,4.99,4.93,79.35,4.81,5.93
Home,5.70,5.58,78.67,5.22,4.83
Life,6.05,5.07,78.66,4.99,5.23


### Annual Premium Format by Product Findings

Annual premium formatting patterns are distributed consistently across all four insurance products.

Approximately 79% of records within each product use the Euro symbol, while the remaining formats — dollar symbols, `EUR` codes, the word `euros`, and plain numeric values — each generally account for around 5% of records.

No formatting pattern is concentrated within a particular insurance product.

Combined with the previously observed similarity in numeric distributions across formatting groups, this provides strong evidence that the different currency representations are formatting inconsistencies rather than genuine differences in currency.

Therefore, during the data-cleaning stage, the currency symbols and textual currency indicators can be standardized while preserving the underlying numeric premium values.

The original raw values will remain unchanged in the data-understanding stage.

### Damage Amount Format Inspection

The `damage_amount` variable represents the estimated monetary value of claim damage but is currently stored as a string.

Its raw formatting patterns are examined before numeric conversion to determine whether the same inconsistencies observed in annual premiums are also present in claim-related monetary variables.

In [53]:
claims["damage_amount"].dropna().astype(str).head(30).to_list()

['15213.03€',
 '2321.55€',
 '1762.45€',
 '3144.06 euros',
 '3715.18€',
 '483.92€',
 '210.13€',
 '13896.38€',
 '561.75€',
 '12836.60€',
 '416.05€',
 '1800.12€',
 '3015.27€',
 '545.72€',
 '207.95€',
 '11538.69€',
 '411.66€',
 '€1814.06',
 '426.22€',
 '$1640.40',
 '5274.03€',
 '301.36€',
 '8979.43€',
 '19682.88€',
 '2924.95€',
 '1589',
 '335.68€',
 '$5263.28',
 '486.98 euros',
 '3048.82€']

In [54]:
damage_str = (
    claims["damage_amount"]
    .astype("string")
    .str.strip()
)

damage_format = pd.Series(
    np.select(
        [
            damage_str.str.contains(r"\$", regex=True, na=False),
            damage_str.str.contains("€", regex=False, na=False),
            damage_str.str.contains(
                r"\bEUR\b",
                case=False,
                regex=True,
                na=False
            ),
            damage_str.str.contains(
                r"\beuros?\b",
                case=False,
                regex=True,
                na=False
            ),
            damage_str.str.fullmatch(
                r"\d+(?:\.\d+)?",
                na=False
            ),
        ],
        [
            "Dollar symbol",
            "Euro symbol",
            "EUR code",
            "Euro word",
            "Plain numeric",
        ],
        default="Other",
    ),
    index=claims.index,
    name="damage_format",
)

damage_format.value_counts(dropna=False)

damage_format
Euro symbol      130
Euro word         10
EUR code           7
Dollar symbol      5
Plain numeric      3
Name: count, dtype: int64

### Damage Amount Format Findings

The `damage_amount` variable exhibits the same type of formatting inconsistency observed in `annual_premium`.

All 155 claim records fall into one of five identifiable formats:

- Euro symbol
- The word `euros`
- `EUR` currency code
- Dollar symbol
- Plain numeric representation

No unexpected formatting patterns were detected.

The Euro symbol is the dominant representation, while only a small number of records use dollar symbols or plain numeric values.

Before standardizing the field, the numeric component will be extracted temporarily and compared across formatting groups to verify that the different currency representations behave like formatting noise rather than genuinely different currencies.

In [55]:
damage_numeric_temp = (
    damage_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

damage_format_analysis = (
    pd.DataFrame(
        {
            "damage_format": damage_format,
            "damage_numeric": damage_numeric_temp,
        }
    )
    .groupby("damage_format")["damage_numeric"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .round(2)
)

damage_format_analysis

,count,mean,median,min,max
damage_format,,,,,
Dollar symbol,5,"3,210.37","2,023.53","1,482.36","5,642.27"
EUR code,7,"6,115.32","4,770.76","1,015.07","19,056.12"
Euro symbol,130,"4,585.07","2,195.16",190.01,"48,459.84"
Euro word,10,"1,715.93","1,768.85",243.80,"3,331.34"
Plain numeric,3,896.00,848.00,251.00,"1,589.00"


In [56]:
print(
    "Unparsed damage values:",
    damage_numeric_temp.isna().sum()
)

Unparsed damage values: 0


### Damage Amount Numeric Distribution Findings

The numeric component of all 155 `damage_amount` values was successfully extracted, with no parsing failures.

The Euro symbol is the dominant formatting pattern, while the remaining formats contain relatively few observations.

Although differences are visible in the descriptive statistics across formatting groups, the sample sizes of the less common formats are too small to conclude that these differences represent genuine currency effects.

The main data-quality issue is therefore the inconsistent textual representation of monetary values rather than an inability to extract their numeric component.

During the data-cleaning stage, the monetary formatting will be standardized while the original raw values remain preserved in the source layer.

### Indemnified Amount Format Inspection

The `indemnified_amount` variable represents the finalized indemnity associated with a claim.

Unlike `damage_amount`, this field contains structurally missing values for claims that have not yet reached a finalized settlement stage.

Only non-missing values are examined for formatting inconsistencies. The original missing values are preserved because previous analysis showed that their absence is related to the claim lifecycle.

In [57]:
indemnified_str = (
    claims["indemnified_amount"]
    .astype("string")
    .str.strip()
)

indemnified_format = pd.Series(
    np.select(
        [
            indemnified_str.str.contains(r"\$", regex=True, na=False),
            indemnified_str.str.contains("€", regex=False, na=False),
            indemnified_str.str.contains(
                r"\bEUR\b",
                case=False,
                regex=True,
                na=False,
            ),
            indemnified_str.str.contains(
                r"\beuros?\b",
                case=False,
                regex=True,
                na=False,
            ),
            indemnified_str.str.fullmatch(
                r"\d+(?:\.\d+)?",
                na=False,
            ),
        ],
        [
            "Dollar symbol",
            "Euro symbol",
            "EUR code",
            "Euro word",
            "Plain numeric",
        ],
        default="Other",
    ),
    index=claims.index,
    name="indemnified_format",
)

In [58]:
indemnified_format = indemnified_format.mask(
    claims["indemnified_amount"].isna(),
    "Missing",
)

In [59]:
indemnified_format.value_counts(dropna=False)

indemnified_format
Euro symbol      71
Missing          65
EUR code          6
Euro word         6
Dollar symbol     4
Plain numeric     3
Name: count, dtype: int64

### Indemnified Amount Format Findings

The `indemnified_amount` variable contains 90 recorded values and 65 missing values.

Among the recorded values, all observations fall into one of five identifiable monetary formatting patterns:

- Euro symbol
- `EUR` currency code
- The word `euros`
- Dollar symbol
- Plain numeric representation

No unexpected formatting patterns were detected.

The 65 missing values are preserved separately because previous analysis showed that their absence is structurally related to the claim lifecycle rather than representing ordinary data-entry errors.

Therefore, monetary formatting and missingness should be treated as two separate data-quality issues:

- Recorded indemnification amounts require numeric standardization.
- Missing indemnification amounts should retain their business meaning and should not automatically be replaced with zero.

In [60]:
indemnified_numeric_temp = (
    indemnified_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

In [61]:
print(
    "Original missing values:",
    claims["indemnified_amount"].isna().sum()
)

print(
    "Parsed missing values:",
    indemnified_numeric_temp.isna().sum()
)

print(
    "Parsing failures among non-missing values:",
    (
        claims["indemnified_amount"].notna()
        & indemnified_numeric_temp.isna()
    ).sum()
)

Original missing values: 65
Parsed missing values: 65
Parsing failures among non-missing values: 0


In [62]:
indemnified_numeric_temp.describe()

count       90.00
mean     2,662.73
std      5,016.24
min          0.00
25%          0.00
50%          0.00
75%      2,417.72
max     28,048.95
Name: indemnified_amount, dtype: float64

### Indemnified Amount Parsing Findings

The numeric component of all non-missing `indemnified_amount` values was successfully extracted.

The parsing process preserved all 65 structurally missing values and introduced no additional missing observations.

Among the 90 recorded indemnification amounts, the median value is zero. This is consistent with the previously observed claim lifecycle:

- Rejected claims contain finalized zero indemnification amounts.
- Closed claims contain finalized indemnification amounts that may be either positive or zero.
- Open, In Progress, and Expert Review claims contain missing indemnification amounts because settlement has not yet been finalized.

This distinction is important for future modeling because a zero indemnification amount and a missing indemnification amount represent different business states.

During the cleaning stage, monetary formatting can therefore be standardized while preserving both explicit zero values and structurally missing values.

## 19. Vehicle Monetary and Numeric Format Inspection

Vehicle-related variables are inspected before numeric conversion.

The `current_value` field represents the monetary value of the insured vehicle but is currently stored as a string. Its formatting patterns are examined to determine whether it contains the same monetary inconsistencies observed in contract and claim data.

In [63]:
vehicles["current_value"].dropna().astype(str).head(30).to_list()

['29567.77€',
 '14873.92€',
 '$8362.30',
 '14849.30€',
 '€14522.08',
 '5563.73€',
 '27063.05 euros',
 '€5771.16',
 '5183.74€',
 '15717.85€',
 '14083.82€',
 '9184.95€',
 '€5874.58',
 '12920.99€',
 '4635.47€',
 '22154.16€',
 '€4868.71',
 '10940.48€',
 '12608.02€',
 '17390.55€',
 '4323.29€',
 '6645.39€',
 '€5836.75',
 '39264.93€',
 '5560.11€',
 '8139.55€',
 '15832.15€',
 '13729.85€',
 '5102.44 euros',
 '12363.13€']

In [64]:
current_value_str = (
    vehicles["current_value"]
    .astype("string")
    .str.strip()
)

current_value_format = pd.Series(
    np.select(
        [
            current_value_str.str.contains(r"\$", regex=True, na=False),
            current_value_str.str.contains("€", regex=False, na=False),
            current_value_str.str.contains(
                r"\bEUR\b",
                case=False,
                regex=True,
                na=False,
            ),
            current_value_str.str.contains(
                r"\beuros?\b",
                case=False,
                regex=True,
                na=False,
            ),
            current_value_str.str.fullmatch(
                r"\d+(?:\.\d+)?",
                na=False,
            ),
        ],
        [
            "Dollar symbol",
            "Euro symbol",
            "EUR code",
            "Euro word",
            "Plain numeric",
        ],
        default="Other",
    ),
    index=vehicles.index,
    name="current_value_format",
)

current_value_format = current_value_format.mask(
    vehicles["current_value"].isna(),
    "Missing",
)

current_value_format.value_counts(dropna=False)

current_value_format
Euro symbol      4267
Euro word         294
EUR code          286
Plain numeric     280
Dollar symbol     263
Name: count, dtype: int64

### Vehicle Current Value Format Findings

All 5,390 `current_value` records fall into one of the previously identified monetary formatting patterns.

The Euro symbol is again the dominant representation, accounting for approximately 79% of vehicle values. Smaller portions of the data use the word `euros`, the `EUR` currency code, dollar symbols, or plain numeric strings.

No missing values or unexpected formatting patterns were detected in this field.

The similarity between the formatting distribution of `current_value` and previously examined monetary variables suggests that monetary formatting inconsistencies are a systematic data-quality characteristic of the dataset.

Before numeric standardization, the numeric component of the field will be extracted temporarily to confirm that all values can be parsed successfully.

In [65]:
current_value_numeric_temp = (
    current_value_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

In [66]:
print(
    "Original missing values:",
    vehicles["current_value"].isna().sum()
)

print(
    "Parsed missing values:",
    current_value_numeric_temp.isna().sum()
)

print(
    "Parsing failures among non-missing values:",
    (
        vehicles["current_value"].notna()
        & current_value_numeric_temp.isna()
    ).sum()
)

Original missing values: 0
Parsed missing values: 0
Parsing failures among non-missing values: 0


In [67]:
current_value_numeric_temp.describe()

count    5,390.00
mean    10,216.81
std      6,616.54
min      4,000.10
25%      5,397.52
50%      8,405.33
75%     12,857.71
max     47,482.80
Name: current_value, dtype: float64

### Vehicle Current Value Parsing Findings

All `current_value` observations were successfully parsed into numeric values.

No missing values were present in the original field, and the parsing process introduced no additional missing observations.

The vehicle value distribution ranges from approximately 4,000 to 47,483, with a mean value higher than the median. This suggests that the distribution may be right-skewed and potentially influenced by higher-value vehicles.

The distribution shape will be investigated further during exploratory data analysis.

From a data-quality perspective, the main issue in `current_value` is inconsistent monetary formatting rather than missing or unparseable values.

### Vehicle Power Format Inspection

The `power` variable is expected to represent vehicle engine power but is currently stored as a string and contains missing values.

Its raw values are inspected before numeric conversion to identify:

- Unit representations
- Textual suffixes or prefixes
- Numeric-only values
- Mixed measurement units
- Invalid or unexpected formats

Missing values are preserved during this inspection.

In [68]:
vehicles["power"].dropna().astype(str).head(40).to_list()

['128 HP',
 '150 HP',
 '175',
 '176hp',
 '122 kW',
 '106 CV',
 '183hp',
 '120 HP',
 '85 kW',
 '99',
 '183hp',
 '192hp',
 '118 CV',
 '139 HP',
 '160 HP',
 '138 CV',
 '97 CV',
 '130 kW',
 '98 CV',
 '80 kW',
 '200 CV',
 '128 CV',
 '108 kW',
 '142',
 '117hp',
 '173',
 '159 CV',
 '163 CV',
 '100',
 '133hp',
 '116 CV',
 '119',
 '99',
 '189 CV',
 '110',
 '141hp',
 '199 HP',
 '73 kW',
 '192 HP',
 '162hp']

In [69]:
vehicles["power"].dropna().astype(str).unique()[:50]

<StringArray>
['128 HP', '150 HP',    '175',  '176hp', '122 kW', '106 CV',  '183hp',
 '120 HP',  '85 kW',     '99',  '192hp', '118 CV', '139 HP', '160 HP',
 '138 CV',  '97 CV', '130 kW',  '98 CV',  '80 kW', '200 CV', '128 CV',
 '108 kW',    '142',  '117hp',    '173', '159 CV', '163 CV',    '100',
  '133hp', '116 CV',    '119', '189 CV',    '110',  '141hp', '199 HP',
  '73 kW', '192 HP',  '162hp', '115 kW',  '197hp', '179 HP',     '90',
 '140 CV',  '152hp',  '124hp', '189 HP',  '168hp', '105 kW', '195 CV',
  '71 kW']
Length: 50, dtype: str

### Vehicle Power Format Findings

The `power` variable contains multiple representations of vehicle engine power.

Initial inspection reveals several formats:

- Horsepower represented as `HP` or `hp`
- Metric horsepower represented as `CV`
- Kilowatts represented as `kW`
- Plain numeric values without an explicit unit

Additional inconsistencies are present in capitalization and spacing.

Unlike monetary formatting inconsistencies, these unit differences cannot be resolved by simply removing the textual suffix. `HP`, `CV`, and `kW` represent different measurement units and must eventually be converted to a common standard.

Plain numeric values require further investigation before a unit is assigned to them.

No conversion is performed at this stage. The frequency and numeric distribution of each power representation will first be examined.

In [70]:
power_str = (
    vehicles["power"]
    .astype("string")
    .str.strip()
)

power_format = pd.Series(
    np.select(
        [
            power_str.str.contains(
                r"\bhp\b",
                case=False,
                regex=True,
                na=False,
            ),
            power_str.str.contains(
                r"\bcv\b",
                case=False,
                regex=True,
                na=False,
            ),
            power_str.str.contains(
                r"\bkw\b",
                case=False,
                regex=True,
                na=False,
            ),
            power_str.str.fullmatch(
                r"\d+(?:\.\d+)?",
                na=False,
            ),
        ],
        [
            "HP",
            "CV",
            "kW",
            "Plain numeric",
        ],
        default="Other",
    ),
    index=vehicles.index,
    name="power_format",
)

power_format = power_format.mask(
    vehicles["power"].isna(),
    "Missing",
)

power_format.value_counts(dropna=False)

power_format
Other            929
Plain numeric    904
Missing          900
kW               895
CV               894
HP               868
Name: count, dtype: int64

### Power Format Classification Review

The initial power-format classification resulted in 929 observations being labeled as `Other`.

However, earlier inspection showed values such as `176hp`, `141hp`, and `197hp`, where the unit is directly attached to the numeric value.

The initial regular expressions relied on word boundaries, which may fail to recognize units attached directly to numbers.

Therefore, the `Other` category is inspected before concluding that these observations represent invalid or unexpected data.

In [71]:
other_power_values = (
    vehicles.loc[
        power_format == "Other",
        "power"
    ]
    .dropna()
    .astype(str)
)

print("Other observations:", len(other_power_values))
print("Unique Other values:", other_power_values.nunique())

other_power_values.unique()[:50]

Other observations: 929
Unique Other values: 111


<StringArray>
['176hp', '183hp', '192hp', '117hp', '133hp', '141hp', '162hp', '197hp',
 '152hp', '124hp', '168hp',  '96hp', '129hp', '132hp', '199hp', '138hp',
 '161hp', '109hp', '166hp', '165hp', '153hp', '158hp',  '95hp', '126hp',
  '90hp', '137hp', '175hp', '114hp', '105hp', '146hp', '195hp', '134hp',
 '191hp', '102hp', '111hp',  '92hp', '188hp', '163hp', '108hp', '115hp',
 '171hp', '127hp', '144hp', '193hp', '169hp', '160hp', '149hp', '164hp',
 '170hp', '181hp']
Length: 50, dtype: str

In [72]:
other_power_values.head(30).to_list()

['176hp',
 '183hp',
 '183hp',
 '192hp',
 '117hp',
 '133hp',
 '141hp',
 '162hp',
 '197hp',
 '152hp',
 '124hp',
 '183hp',
 '168hp',
 '96hp',
 '129hp',
 '132hp',
 '199hp',
 '138hp',
 '161hp',
 '109hp',
 '166hp',
 '165hp',
 '197hp',
 '153hp',
 '176hp',
 '158hp',
 '95hp',
 '126hp',
 '90hp',
 '137hp']

### Power Format Classification Findings

Inspection of the initially classified `Other` values revealed that they are valid horsepower observations such as `176hp`, `183hp`, and `192hp`.

These records were incorrectly classified because the initial regular expression relied on word boundaries around the unit. When the unit is directly attached to the numeric value, such as `176hp`, the pattern does not detect it correctly.

Therefore, the format classification rule is revised to allow optional whitespace between the numeric value and the measurement unit.

This is a parsing-rule issue rather than a data-quality issue.

In [73]:
power_format = pd.Series(
    np.select(
        [
            power_str.str.fullmatch(
                r"\d+(?:\.\d+)?\s*hp",
                case=False,
                na=False,
            ),
            power_str.str.fullmatch(
                r"\d+(?:\.\d+)?\s*cv",
                case=False,
                na=False,
            ),
            power_str.str.fullmatch(
                r"\d+(?:\.\d+)?\s*kw",
                case=False,
                na=False,
            ),
            power_str.str.fullmatch(
                r"\d+(?:\.\d+)?",
                na=False,
            ),
        ],
        [
            "HP",
            "CV",
            "kW",
            "Plain numeric",
        ],
        default="Other",
    ),
    index=vehicles.index,
    name="power_format",
)

power_format = power_format.mask(
    vehicles["power"].isna(),
    "Missing",
)

power_format.value_counts(dropna=False)

power_format
HP               1797
Plain numeric     904
Missing           900
kW                895
CV                894
Name: count, dtype: int64

In [74]:
print(
    "Remaining Other values:",
    (power_format == "Other").sum()
)

Remaining Other values: 0


### Revised Vehicle Power Format Findings

After revising the format-detection rules, all non-missing `power` values were successfully classified.

The vehicle power field contains four identifiable representations:

- `HP`
- `CV`
- `kW`
- Plain numeric values without an explicit unit

In addition, 900 vehicle records have missing power information.

No observations remain in the `Other` category, confirming that the earlier unclassified values were caused by an overly restrictive parsing rule rather than invalid source data.

The next step is to compare the numeric distributions across unit groups, particularly the plain numeric observations, before deciding how the variable should be standardized.

In [75]:
power_numeric_temp = (
    power_str
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
    .astype(float)
)

In [76]:
power_distribution = (
    pd.DataFrame(
        {
            "power_format": power_format,
            "power_numeric": power_numeric_temp,
        }
    )
    .query("power_format != 'Missing'")
    .groupby("power_format")["power_numeric"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .round(2)
)

power_distribution

,count,mean,median,min,max
power_format,,,,,
CV,894,146.13,146.00,90.00,200.00
HP,1797,145.11,145.00,90.00,200.00
Plain numeric,904,144.27,143.00,90.00,200.00
kW,895,107.96,108.00,66.00,148.00


In [77]:
print(
    "Original missing values:",
    vehicles["power"].isna().sum()
)

print(
    "Parsed missing values:",
    power_numeric_temp.isna().sum()
)

print(
    "Parsing failures among non-missing values:",
    (
        vehicles["power"].notna()
        & power_numeric_temp.isna()
    ).sum()
)

Original missing values: 900
Parsed missing values: 900
Parsing failures among non-missing values: 0


### Vehicle Power Distribution Findings

All non-missing `power` values were successfully parsed, and no additional missing values were introduced during numeric extraction.

The numeric distributions provide an important insight into the underlying measurement structure:

- HP values have a mean of approximately 145.
- CV values have a mean of approximately 146.
- Plain numeric values have a mean of approximately 144.
- kW values have a lower raw mean of approximately 108 because they use a different measurement unit.

When expressed in horsepower terms, the average kW value is approximately equivalent to the HP, CV, and plain numeric groups.

This strongly suggests that the different representations describe the same underlying vehicle-power distribution rather than fundamentally different groups of vehicles.

Plain numeric observations also closely match the HP and CV distributions, suggesting that they may represent power values with a missing unit label.

However, the unit of plain numeric observations will not be assigned solely based on distributional similarity. Additional validation will be performed before defining the final standardization rule.

The 900 original missing power values remain unchanged.

### Temporary Power Standardization

To evaluate whether the different power representations describe a common underlying distribution, explicitly labeled HP, CV, and kW values are temporarily converted to horsepower.

The conversion is performed only for exploratory validation. The original `power` column remains unchanged.

Plain numeric observations are intentionally left unconverted because their measurement unit has not yet been formally established.

In [78]:
power_analysis = pd.DataFrame(
    {
        "power_format": power_format,
        "power_numeric": power_numeric_temp,
    }
)

power_analysis["power_hp_temp"] = np.select(
    [
        power_analysis["power_format"].eq("HP"),
        power_analysis["power_format"].eq("CV"),
        power_analysis["power_format"].eq("kW"),
    ],
    [
        power_analysis["power_numeric"],
        power_analysis["power_numeric"] * 0.98632,
        power_analysis["power_numeric"] * 1.34102,
    ],
    default=np.nan,
)

In [79]:
power_standardized_summary = (
    power_analysis
    .query("power_format in ['HP', 'CV', 'kW']")
    .groupby("power_format")["power_hp_temp"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .round(2)
)

power_standardized_summary

,count,mean,median,min,max
power_format,,,,,
CV,894,144.14,144.00,88.77,197.26
HP,1797,145.11,145.00,90.00,200.00
kW,895,144.77,144.83,88.51,198.47


In [80]:
print(
    "Plain numeric mean:",
    round(
        power_analysis.loc[
            power_analysis["power_format"] == "Plain numeric",
            "power_numeric"
        ].mean(),
        2
    )
)

print(
    "Plain numeric median:",
    round(
        power_analysis.loc[
            power_analysis["power_format"] == "Plain numeric",
            "power_numeric"
        ].median(),
        2
    )
)

Plain numeric mean: 144.27
Plain numeric median: 143.0


### Temporary Power Standardization Findings

After temporarily converting explicitly labeled power values to horsepower, the HP, CV, and kW groups exhibit almost identical distributions.

The standardized mean values are approximately:

- 144.14 HP for values originally expressed in CV
- 145.11 HP for values originally expressed in HP
- 144.77 HP for values originally expressed in kW

Plain numeric observations have a mean of 144.27 and a median of 143, which closely matches the standardized distributions of the explicitly labeled groups.

This provides strong evidence that the different representations describe the same underlying vehicle-power variable.

However, the exact unit of plain numeric observations cannot be distinguished with certainty from distributional evidence alone because HP and CV are numerically very similar.

Therefore, before assigning a unit to plain numeric observations during data cleaning, an additional validation will be performed using vehicle characteristics such as brand, model, and year.

### Plain Numeric Power Validation by Vehicle Characteristics

Plain numeric power values are compared with explicitly labeled power values for similar vehicles.

Vehicle records sharing the same brand, model, and model year are grouped together to determine whether plain numeric observations align with standardized power values from explicitly labeled records.

This provides an additional validation before assigning a measurement unit to records where the unit is missing.

In [81]:
power_vehicle_analysis = vehicles[
    [
        "brand",
        "model",
        "year",
        "power",
    ]
].copy()

power_vehicle_analysis["power_format"] = power_format
power_vehicle_analysis["power_numeric"] = power_numeric_temp

In [82]:
power_vehicle_analysis["power_hp_temp"] = np.select(
    [
        power_vehicle_analysis["power_format"].eq("HP"),
        power_vehicle_analysis["power_format"].eq("CV"),
        power_vehicle_analysis["power_format"].eq("kW"),
    ],
    [
        power_vehicle_analysis["power_numeric"],
        power_vehicle_analysis["power_numeric"] * 0.98632,
        power_vehicle_analysis["power_numeric"] * 1.34102,
    ],
    default=np.nan,
)

In [83]:
comparable_power_groups = (
    power_vehicle_analysis
    .dropna(subset=["brand", "model", "year"])
    .groupby(["brand", "model", "year"])
    .filter(
        lambda x: (
            x["power_format"].eq("Plain numeric").any()
            and x["power_format"].isin(["HP", "CV", "kW"]).any()
        )
    )
)

print(
    "Comparable observations:",
    len(comparable_power_groups)
)

print(
    "Comparable vehicle groups:",
    comparable_power_groups[
        ["brand", "model", "year"]
    ].drop_duplicates().shape[0]
)

Comparable observations: 4924
Comparable vehicle groups: 181


### Comparable Vehicle Group Findings

A total of 181 distinct brand-model-year groups contain both plain numeric power values and explicitly labeled power values.

These groups represent 4,924 vehicle observations, providing a sufficiently large basis for validating the interpretation of unitless power values.

The next step compares each plain numeric power observation with the standardized horsepower level observed among explicitly labeled records for the same brand, model, and model year.

In [84]:
labeled_power_reference = (
    power_vehicle_analysis
    .loc[
        power_vehicle_analysis["power_format"].isin(
            ["HP", "CV", "kW"]
        )
    ]
    .dropna(
        subset=[
            "brand",
            "model",
            "year",
            "power_hp_temp",
        ]
    )
    .groupby(
        ["brand", "model", "year"]
    )["power_hp_temp"]
    .median()
    .rename("reference_hp")
    .reset_index()
)

labeled_power_reference.head()

,brand,model,year,reference_hp
0,BMW,Serie1,"2,010.00",97.00
1,BMW,Serie1,"2,011.00",140.00
2,BMW,Serie1,"2,012.00",135.92
3,BMW,Serie1,"2,013.00",111.00
4,BMW,Serie1,"2,014.00",127.52


In [85]:
plain_power_comparison = (
    power_vehicle_analysis
    .loc[
        power_vehicle_analysis["power_format"]
        .eq("Plain numeric"),
        [
            "brand",
            "model",
            "year",
            "power_numeric",
        ],
    ]
    .merge(
        labeled_power_reference,
        on=["brand", "model", "year"],
        how="inner",
        validate="many_to_one",
    )
)

plain_power_comparison["absolute_difference"] = (
    plain_power_comparison["power_numeric"]
    - plain_power_comparison["reference_hp"]
).abs()

plain_power_comparison["pct_difference"] = (
    plain_power_comparison["absolute_difference"]
    / plain_power_comparison["reference_hp"]
    * 100
)

plain_power_comparison.head(10)

,brand,model,year,power_numeric,reference_hp,absolute_difference,pct_difference
0,Peugeot,208,"2,020.00",175.00,131.00,44.00,33.59
1,Peugeot,3008,"2,021.00",99.00,168.00,69.00,41.07
2,Volkswagen,Polo,"2,023.00",142.00,159.51,17.51,10.97
3,Mercedes,ClasseA,"2,024.00",173.00,142.00,31.00,21.83
4,Peugeot,308,"2,018.00",100.00,146.00,46.00,31.51
5,Peugeot,208,"2,021.00",119.00,137.10,18.10,13.20
6,Renault,Megane,"2,020.00",110.00,153.87,43.87,28.51
7,Renault,Captur,"2,022.00",90.00,141.00,51.00,36.17
8,Peugeot,3008,"2,010.00",110.00,112.50,2.50,2.22
9,Renault,Megane,"2,019.00",180.00,144.00,36.00,25.00


In [86]:
plain_power_comparison[
    [
        "absolute_difference",
        "pct_difference",
    ]
].describe()

,absolute_difference,pct_difference
count,858.00,858.00
mean,29.48,20.35
std,17.87,12.57
min,0.00,0.00
25%,14.00,9.56
50%,28.34,19.42
75%,44.00,30.12
max,78.05,66.22


In [87]:
print(
    "Comparable plain numeric observations:",
    len(plain_power_comparison)
)

print(
    "Plain numeric observations within 5% of reference HP:",
    (
        plain_power_comparison["pct_difference"] <= 5
    ).sum()
)

print(
    "Share within 5%:",
    round(
        (
            plain_power_comparison["pct_difference"] <= 5
        ).mean()
        * 100,
        2,
    ),
    "%"
)

Comparable plain numeric observations: 858
Plain numeric observations within 5% of reference HP: 100
Share within 5%: 11.66 %


## 20. Date Format Inspection

Date-related variables are inspected before conversion to datetime data types.

The objective of this step is to identify:

- Date formatting patterns
- Potential parsing inconsistencies
- Missing date values
- Invalid or suspicious date relationships

No date conversion or correction is performed at this stage.

In [88]:
date_columns = {
    "contracts.start_date": contracts["start_date"],
    "contracts.end_date": contracts["end_date"],
    "claims.occurrence_date": claims["occurrence_date"],
    "claims.declaration_date": claims["declaration_date"],
}

In [89]:
for name, series in date_columns.items():
    print(f"\n{name.upper()}")
    print("-" * 50)

    print(
        series
        .dropna()
        .astype(str)
        .head(20)
        .to_list()
    )


CONTRACTS.START_DATE
--------------------------------------------------
['11/08/2023', '2025-08-12', '2025-06-14', '04/17/2023', '2025-03-02', '2023-06-10', '2024-04-18', '14-07-2023', '2024-12-13', '2024-01-03', '2024-10-30', '2024-03-16', '2025-01-08', '2024-04-28', '2024-06-16', '2024-07-18', '16-06-2024', '2023-09-19', '07/11/2025', '2025-01-01']

CONTRACTS.END_DATE
--------------------------------------------------
['2024-09-08', '2026-08-15', '2026-06-30', '2024-04-20', '2026-02-26', '2024-06-05', '2025-03-15', '2024-07-13', '2025-12-24', '2025-01-13', '14-12-2025', '2025-04-20', '2026-01-21', '20-04-2025', '14-06-2025', '17/08/2025', '2025-04-20', '2024-08-05', '2026-07-23', '2026-01-12']

CLAIMS.OCCURRENCE_DATE
--------------------------------------------------
['26-11-2023', '2025-08-26', '2024-10-25', '2024-04-16', '08/03/2025', '2024-02-10', '2025-12-01', '2024-08-12', '2025-07-22', '02/11/2025', '2025-06-11', '2023-09-04', '2025-08-12', '2025-05-31', '2025-04-02', '2025-05

In [90]:
date_missing_summary = pd.DataFrame(
    {
        "column": date_columns.keys(),
        "missing_count": [
            series.isna().sum()
            for series in date_columns.values()
        ],
        "missing_pct": [
            series.isna().mean() * 100
            for series in date_columns.values()
        ],
    }
)

date_missing_summary

,column,missing_count,missing_pct
0,contracts.start_date,0,0.00
1,contracts.end_date,0,0.00
2,claims.occurrence_date,0,0.00
3,claims.declaration_date,0,0.00


### Date Format Findings

No missing values were detected in the four date-related variables.

However, the raw date fields contain multiple formatting conventions, including:

- ISO-style dates such as `2025-08-12`
- Slash-separated dates such as `11/08/2023`
- Day-first dash-separated dates such as `26-11-2023`

Some slash-separated dates may be ambiguous because both the day and month can be less than or equal to 12.

Therefore, date parsing should be handled carefully during the data-cleaning stage rather than assuming a single date format across all records.

In [91]:
date_parsing_summary = []

for name, series in date_columns.items():
    parsed = pd.to_datetime(
        series,
        format="mixed",
        errors="coerce",
    )

    date_parsing_summary.append(
        {
            "column": name,
            "rows": len(series),
            "parsing_failures": parsed.isna().sum(),
            "parsing_failure_pct": parsed.isna().mean() * 100,
        }
    )

date_parsing_summary = pd.DataFrame(date_parsing_summary)

date_parsing_summary

,column,rows,parsing_failures,parsing_failure_pct
0,contracts.start_date,15000,0,0.00
1,contracts.end_date,15000,0,0.00
2,claims.occurrence_date,155,0,0.00
3,claims.declaration_date,155,0,0.00


### Date Parsing Findings

All date-related variables can be technically parsed without introducing missing values.

No parsing failures were detected in:

- `contracts.start_date`
- `contracts.end_date`
- `claims.occurrence_date`
- `claims.declaration_date`

However, successful parsing does not guarantee that ambiguous date representations have been interpreted correctly.

Some slash-separated dates may support more than one valid day-month interpretation. Therefore, date standardization will be handled carefully during the data-cleaning stage and validated using logical relationships such as contract duration and claim declaration timing.

## 21. Categorical Variable Inspection

Categorical variables are reviewed to identify:

- The number of distinct categories
- Category naming inconsistencies
- Potential spelling or capitalization issues
- Unexpected values
- Variables with high cardinality

This inspection will guide category standardization during the data-cleaning stage.

In [92]:
categorical_columns = {
    "contracts": [
        "product",
        "status",
        "city_postal",
        "risk_zone",
        "channel",
        "csp",
        "gender",
    ],
    "claims": [
        "claim_type",
        "status",
        "expert_id",
        "liability",
    ],
    "vehicles": [
        "brand",
        "model",
        "fuel_type",
        "color",
        "usage",
    ],
}

categorical_summary = []

for dataset_name, columns in categorical_columns.items():
    df = datasets[dataset_name]

    for column in columns:
        categorical_summary.append(
            {
                "dataset": dataset_name,
                "column": column,
                "unique_values": df[column].nunique(dropna=True),
                "missing_count": df[column].isna().sum(),
                "missing_pct": df[column].isna().mean() * 100,
            }
        )

categorical_summary = pd.DataFrame(categorical_summary)

categorical_summary

,dataset,column,unique_values,missing_count,missing_pct
0,contracts,product,4,0,0.00
1,contracts,status,5,0,0.00
2,contracts,city_postal,7,0,0.00
3,contracts,risk_zone,3,0,0.00
4,contracts,channel,4,0,0.00
5,contracts,csp,7,1772,11.81
6,contracts,gender,4,3096,20.64
7,claims,claim_type,7,0,0.00
8,claims,status,5,0,0.00
9,claims,expert_id,28,38,24.52


### Categorical Variable Findings

Most categorical variables have relatively low cardinality, which makes them suitable for descriptive analysis, dashboarding, and potential use in machine learning models.

Notable observations include:

- Insurance products are represented by four categories.
- Contract and claim statuses each contain five distinct values.
- Risk zone, sales channel, fuel type, vehicle usage, and similar operational variables have low cardinality.
- Vehicle brand contains five categories and vehicle model contains fifteen categories.
- Missing values are present in `csp`, `gender`, `expert_id`, `liability`, and `color`.
- `expert_id` contains 28 distinct identifiers and should be treated primarily as an operational identifier rather than a general categorical feature.
- Claim-related operational variables such as `expert_id` must also be evaluated for potential data leakage before being used in predictive models.

The actual category labels will now be inspected for inconsistencies in spelling, capitalization, spacing, or representation.

In [93]:
low_cardinality_columns = {
    "contracts": [
        "product",
        "status",
        "city_postal",
        "risk_zone",
        "channel",
        "csp",
        "gender",
    ],
    "claims": [
        "claim_type",
        "status",
        "liability",
    ],
    "vehicles": [
        "brand",
        "model",
        "fuel_type",
        "color",
        "usage",
    ],
}

for dataset_name, columns in low_cardinality_columns.items():
    df = datasets[dataset_name]

    print("\n" + "=" * 70)
    print(dataset_name.upper())
    print("=" * 70)

    for column in columns:
        values = (
            df[column]
            .dropna()
            .astype(str)
            .sort_values()
            .unique()
        )

        print(f"\n{column}:")
        print(values)


CONTRACTS

product:
<StringArray>
['Auto', 'Health', 'Home', 'Life']
Length: 4, dtype: str

status:
<StringArray>
['Active', 'Cancelled', 'Expired', 'Renewed', 'Suspended']
Length: 5, dtype: str

city_postal:
<StringArray>
[ 'Bordeaux_33000',     'Dijon_21000',      'Lyon_69000', 'Marseille_13000',
    'Nantes_44000',     'Paris_75001',  'Toulouse_31000']
Length: 7, dtype: str

risk_zone:
<StringArray>
['High', 'Low', 'Medium']
Length: 3, dtype: str

channel:
<StringArray>
['Agency', 'Broker', 'Phone', 'Web']
Length: 4, dtype: str

csp:
<StringArray>
[     'Employee',       'Manager',       'Retired', 'Self_employed',
       'Student',    'Unemployed',        'Worker']
Length: 7, dtype: str

gender:
<StringArray>
['F', 'Female', 'M', 'Male']
Length: 4, dtype: str

CLAIMS

claim_type:
<StringArray>
[   'Collision',         'Fire', 'Glass_damage',        'Storm',
        'Theft',    'Vandalism', 'Water_damage']
Length: 7, dtype: str

status:
<StringArray>
['Closed', 'Expert_review', 'In

### Categorical Value Findings

The categorical value inspection shows that most categorical variables are consistently represented without major capitalization, spelling, or whitespace issues.

The main inconsistency is observed in the `gender` variable, where the same categories are represented using both abbreviated and full labels:

- `F` and `Female`
- `M` and `Male`

These values should be standardized during data cleaning.

Additional structural observations include:

- `city_postal` combines city and postal-code information in a single field and may be separated into two variables during preprocessing.
- Variables such as `claim_type`, `liability`, and `csp` use consistent underscore-based category labels.
- Vehicle brand, model, fuel type, color, and usage categories appear internally consistent.
- `expert_id` should be treated as an operational identifier rather than a general categorical feature.

No categorical transformations are applied during the data-understanding stage.

## 22. Basic Validity Checks

A small set of logical validity checks is performed to identify potentially inconsistent records before data cleaning.

The checks focus on:

- Contract date consistency
- Claim date consistency
- Client age ranges
- Vehicle year ranges
- Previous claim values

These checks are intended to identify suspicious observations rather than automatically correct or remove them.

In [94]:
start_date_temp = pd.to_datetime(
    contracts["start_date"],
    format="mixed",
    errors="coerce",
)

end_date_temp = pd.to_datetime(
    contracts["end_date"],
    format="mixed",
    errors="coerce",
)

occurrence_date_temp = pd.to_datetime(
    claims["occurrence_date"],
    format="mixed",
    errors="coerce",
)

declaration_date_temp = pd.to_datetime(
    claims["declaration_date"],
    format="mixed",
    errors="coerce",
)

In [95]:
validity_summary = pd.Series(
    {
        "contracts_end_before_start": (
            end_date_temp < start_date_temp
        ).sum(),

        "claims_declared_before_occurrence": (
            declaration_date_temp < occurrence_date_temp
        ).sum(),

        "client_age_below_18": (
            contracts["client_age"] < 18
        ).sum(),

        "client_age_above_100": (
            contracts["client_age"] > 100
        ).sum(),

        "vehicle_year_before_1950": (
            vehicles["year"] < 1950
        ).sum(),

        "vehicle_year_above_2026": (
            vehicles["year"] > 2026
        ).sum(),

        "previous_claims_negative": (
            vehicles["previous_claims"] < 0
        ).sum(),
    },
    name="invalid_record_count",
)

validity_summary

contracts_end_before_start            0
claims_declared_before_occurrence    63
client_age_below_18                   0
client_age_above_100                  0
vehicle_year_before_1950              0
vehicle_year_above_2026               0
previous_claims_negative              0
Name: invalid_record_count, dtype: int64

### Basic Validity Check Findings

Most basic validity checks did not reveal logical inconsistencies:

- No contracts have an end date earlier than their start date.
- No client ages fall outside the expected range of 18 to 100.
- No vehicle years fall outside the defined validity range.
- No negative values were observed in `previous_claims`.

However, 63 claim records appear to have a declaration date earlier than the corresponding occurrence date.

Given the previously identified mixture of date formats, this result should not immediately be interpreted as a source-data error. Ambiguous day-month representations may have been parsed incorrectly during the temporary datetime conversion.

Therefore, these records require additional inspection before the claim date relationship can be classified as invalid.

In [96]:
claim_date_check = claims[
    [
        "claim_id",
        "occurrence_date",
        "declaration_date",
    ]
].copy()

claim_date_check["occurrence_parsed"] = occurrence_date_temp
claim_date_check["declaration_parsed"] = declaration_date_temp

suspicious_claim_dates = claim_date_check.loc[
    claim_date_check["declaration_parsed"]
    < claim_date_check["occurrence_parsed"]
].copy()

print(
    "Suspicious claim date records:",
    len(suspicious_claim_dates)
)

suspicious_claim_dates.head(20)

Suspicious claim date records: 63


,claim_id,occurrence_date,declaration_date,occurrence_parsed,declaration_parsed
0,CLM_0000001,26-11-2023,2023-10-02,2023-11-26,2023-10-02
2,CLM_0000003,2024-10-25,2024-09-26,2024-10-25,2024-09-26
4,CLM_0000005,08/03/2025,2025-04-12,2025-08-03,2025-04-12
8,CLM_0000009,2025-07-22,2025-06-16,2025-07-22,2025-06-16
10,CLM_0000011,2025-06-11,2025-06-10,2025-06-11,2025-06-10
11,CLM_0000012,2023-09-04,2023-07-31,2023-09-04,2023-07-31
12,CLM_0000013,2025-08-12,2025-06-28,2025-08-12,2025-06-28
17,CLM_0000018,2025-04-01,2025-03-09,2025-04-01,2025-03-09
18,CLM_0000019,2024-07-18,2024-06-17,2024-07-18,2024-06-17
19,CLM_0000020,26-06-2023,01/07/2023,2023-06-26,2023-01-07


### Basic Validity Check Findings

Most basic validity checks did not reveal logical inconsistencies:

- No contracts have an end date earlier than their start date.
- No client ages fall outside the expected range of 18 to 100.
- No vehicle years fall outside the defined validity range.
- No negative values were observed in `previous_claims`.

However, 63 out of 155 claim records have a parsed declaration date earlier than their occurrence date.

Further inspection shows that this issue is not caused solely by ambiguous date parsing. Some records use unambiguous ISO-style dates and still violate the expected chronological relationship between claim occurrence and declaration.

Therefore, the claim date fields contain two separate data-quality issues:

- Mixed and potentially ambiguous date formats
- Logical inconsistencies where `declaration_date < occurrence_date`

These records will require explicit treatment during the data-cleaning stage rather than automatic correction during data understanding.

## 23. Data Understanding Summary

The initial data-understanding stage identified the structure, relationships, and major data-quality characteristics of the Insurance Analytics Platform dataset.

### Dataset Structure

The project contains three related datasets:

- `contracts`: 15,000 insurance contracts
- `claims`: 155 claim records
- `vehicles`: 5,390 vehicle records

`contract_id` is unique in the contracts dataset and serves as the central relationship key.

All `contract_id` values in the claims and vehicles datasets correspond to valid contracts, and no orphan records were detected.

### Portfolio Structure

The insurance portfolio consists of four products:

- Auto
- Home
- Life
- Health

Vehicle information is exclusively associated with Auto insurance policies and is available for approximately 85% of Auto contracts.

Claims are strongly concentrated in the Auto portfolio, while no claim observations are available for Life or Health policies in the current dataset.

### Missing Data

Missing values were identified in several fields, including:

- `client_age`
- `csp`
- `gender`
- `indemnified_amount`
- `expert_id`
- `liability`
- `year`
- `power`
- `color`
- `previous_claims`

Some missing values have a clear business interpretation.

In particular, missing `indemnified_amount` values are strongly related to claim lifecycle status and should not automatically be replaced with zero.

### Monetary Variables

Several monetary fields are stored as strings using inconsistent representations:

- Currency symbols
- Currency codes
- Currency words
- Plain numeric values

Affected variables include:

- `annual_premium`
- `damage_amount`
- `indemnified_amount`
- `current_value`

The numeric components of these fields can be successfully extracted, but formatting must be standardized during data cleaning.

### Vehicle Power

The `power` variable contains multiple measurement representations:

- HP
- CV
- kW
- Unitless numeric values

Explicitly labeled HP, CV, and kW values can be standardized to a common unit.

Unitless values require a documented assumption or additional treatment during preprocessing.

### Categorical Variables

Most categorical variables are internally consistent and have relatively low cardinality.

The primary categorical inconsistency identified is the `gender` variable:

- `F` / `Female`
- `M` / `Male`

These categories should be standardized.

The `city_postal` variable also combines two attributes and may be separated into city and postal-code fields.

### Date Variables

Date fields contain mixed formatting conventions.

Although all date values are technically parseable, some formats are ambiguous.

Additionally, 63 claim records contain a declaration date earlier than the occurrence date, including several unambiguous cases. These observations represent an important data-quality issue that must be addressed during cleaning.

### Modeling Considerations

The initial analysis also revealed several implications for future machine learning tasks:

- Claim occurrence is highly imbalanced.
- Auto insurance is the strongest candidate for claim prediction because it contains the majority of positive claim observations and additional vehicle-level features.
- Claim operational variables generated after claim occurrence must be excluded from claim-prediction models to prevent data leakage.
- Customer-level RFM analysis may require adaptation because `client_id` is unique for every contract in the current dataset.
- Claim severity modeling will need to distinguish between explicit zero indemnification and structurally missing indemnification values.

The next stage of the project will focus on systematic data cleaning and validation while preserving the business meaning identified during this analysis.